# Collapsing the hand construction: separate factors $\to$ combined circuits

The hand-built circuit $W^\star$ is specified as **eight factor matrices** — $W_Q, W_K, W_V, W_O$
for each of the two layers. The `combined` parametrization instead stores **four circuit
matrices**, $W_{qk}$ and $W_{ov}$ per layer.

This notebook does two things:

1. forms the collapse $\;W_{qk}=W_Q^\top W_K,\;\;W_{ov}=W_V^\top W_O^\top\;$ from the eight
   hand-set factors, and installs it in a `combined` model;
2. checks that this model is **bit-for-bit the same function** as a `separate` model holding the
   eight factors directly.

The second point is not optional. The collapse is an algebraic identity, so any discrepancy
beyond float32 rounding would mean the construction, the architecture, or the identity itself is
wrong.

### Why the identity holds

`nn.Linear(x)` computes $x W^\top$. In the **separate** block,

$$Q = x W_Q^\top,\quad K = x W_K^\top \;\Longrightarrow\;
QK^\top = x\,W_Q^\top W_K\,x^\top ,$$

$$\text{attn} = W_O\bigl(\text{softmax}\cdot xW_V^\top\bigr)
= \text{softmax}\cdot x\,\bigl(W_V^\top W_O^\top\bigr).$$

The **combined** block computes $x W_{qk} x^\top$ and $\text{softmax}\cdot x W_{ov}$ directly.
Matching the two gives exactly

$$W_{qk}=W_Q^\top W_K,\qquad W_{ov}=W_V^\top W_O^\top .$$

Both blocks divide the score by the same $\sqrt{\text{output\_dim}}$, so nothing else differs.

## 1. Setup and the eight hand-set factors

Shapes follow `nn.Linear` storage, `[out, in]`. Layer 1 maps $108\to108$; layer 2 maps
$216\to108$ except $W_O$, which is $108\to108$.

In [1]:
import importlib.util, os
import numpy as np
import torch
import torch.nn.functional as F

F14 = os.path.join('..', '14. elimination final convergence confusion')
_s = importlib.util.spec_from_file_location('tw', os.path.join(F14, 'train_with_equiprobable.py'))
TW = importlib.util.module_from_spec(_s); _s.loader.exec_module(TW)
Model = TW.SimplifiedTransformerMLP

V, VS, T = 8, 4, 100
NN, D, IN2 = V - VS, V + T, 2 * (V + T)
EPS, K_LOG = TW.CFG['freeze_delta'], TW.CFG['freeze_hidden_k']
TS, TE = TW.CFG['token_start'], T
DEV = torch.device('cpu')
DATA = os.path.join(F14, 'dataset')

def factors(dev=DEV):
    Wq1 = torch.eye(D, device=dev) * (D ** 0.5)
    M = torch.zeros(D, D, device=dev)
    for k in range(VS, V):
        M[k, :] = -1e9
    for tq in range(T):
        for tk in range(T):
            M[V + tk, V + tq] = -1e9 if tq == tk else -20.0 * (tq - tk)
    Wk1 = M.t()
    Wv1 = torch.zeros(D, D, device=dev); Wv1[:V, :V] = torch.eye(V, device=dev)
    Wo1 = torch.eye(D, device=dev)

    Wq2 = torch.zeros(D, IN2, device=dev)
    Wq2[:VS, :VS] = torch.eye(VS, device=dev) * (D ** 0.5) * 1000.0
    Wq2[:VS, D:D + VS] = torch.eye(VS, device=dev) * (D ** 0.5) * 50.0
    for v in range(V):
        Wq2[VS, v] = (D ** 0.5) * 10.0
    Wk2 = torch.zeros(D, IN2, device=dev); Wk2[:V, D:D + V] = torch.eye(V, device=dev)
    for v in range(VS, V):
        Wk2[VS, v] = -50000.0
    Wv2 = torch.zeros(D, IN2, device=dev); Wv2[:V, :V] = torch.eye(V, device=dev)
    Wo2 = torch.eye(D, device=dev)
    return dict(Wq1=Wq1, Wk1=Wk1, Wv1=Wv1, Wo1=Wo1, Wq2=Wq2, Wk2=Wk2, Wv2=Wv2, Wo2=Wo2)

FA = factors()
for n, m in FA.items():
    print(f'{n}  shape {tuple(m.shape)}   nnz {int((m != 0).sum()):6d}   '
          f'max|.| {float(m.abs().max()):.4g}')

Wq1  shape (108, 108)   nnz    108   max|.| 10.39
Wk1  shape (108, 108)   nnz  10432   max|.| 1e+09
Wv1  shape (108, 108)   nnz      8   max|.| 1
Wo1  shape (108, 108)   nnz    108   max|.| 1
Wq2  shape (108, 216)   nnz     16   max|.| 1.039e+04
Wk2  shape (108, 216)   nnz     12   max|.| 5e+04
Wv2  shape (108, 216)   nnz      8   max|.| 1
Wo2  shape (108, 108)   nnz    108   max|.| 1


## 2. The collapse

$W_{qk}=W_Q^\top W_K$ and $W_{ov}=W_V^\top W_O^\top$, per layer.

In [2]:
def collapse(fa):
    return dict(
        Wqk1=fa['Wq1'].t() @ fa['Wk1'],
        Wov1=fa['Wv1'].t() @ fa['Wo1'].t(),
        Wqk2=fa['Wq2'].t() @ fa['Wk2'],
        Wov2=fa['Wv2'].t() @ fa['Wo2'].t(),
    )

CB = collapse(FA)
for n, m in CB.items():
    vals = sorted({round(float(v), 3) for v in m[m != 0]})
    print(f'{n}  shape {tuple(m.shape)}   nnz {int((m != 0).sum()):6d}   '
          f'distinct {len(vals)}   {vals[:3]} ... {vals[-3:]}')

Wqk1  shape (108, 108)   nnz  10432   distinct 199   [-10392304640.0, -20576.764, -20368.916] ... [20161.07, 20368.916, 20576.764]
Wov1  shape (108, 108)   nnz      8   distinct 1   [1.0] ... [1.0]
Wqk2  shape (216, 216)   nnz     48   distinct 4   [-5196152.5, 103.923, 519.615] ... [103.923, 519.615, 10392.305]
Wov2  shape (216, 108)   nnz      8   distinct 1   [1.0] ... [1.0]


## 3. Both models

`separate` holds the eight factors; `combined` holds the four collapsed circuits. Everything
downstream — the frozen log FFN, the identity selector, the noise bias — is identical in both.

In [3]:
def log_core(dev):
    a, b, w, c, _ = TW.fit_log_core(EPS, K_LOG, TW.CFG['freeze_fit_fmax'],
                                    TW.CFG['freeze_fit_cgain'])
    return a.to(dev), b.to(dev), w.to(dev), c

def noise_bias(alpha):
    return -30.0 if alpha >= 1.0 else float(np.log((1 + VS * EPS) * (1 - alpha) / (alpha * NN)))

def readout_(m, alpha, dev):
    a, b, w, c = log_core(dev)
    with torch.no_grad():
        m.ff1.weight.zero_(); m.ff1.bias.zero_()
        m.ff2.weight.zero_(); m.ff2.bias.zero_()
        for v in range(VS):
            r = slice(v * K_LOG, (v + 1) * K_LOG)
            m.ff1.weight[r, m.in2 + v] = a
            m.ff1.bias[r] = b
            m.ff2.weight[m.in2 + v, r] = w
            m.ff2.bias[m.in2 + v] = c
        m.fc_out.weight.zero_(); m.fc_out.bias.zero_()
        m.fc_out.weight[:VS, m.in2:m.in2 + VS] = torch.eye(VS, device=dev)
        m.fc_out.bias[VS:] = noise_bias(alpha)
    for p in m.parameters():
        p.requires_grad_(False)
    return m.eval()

def build_separate(alpha, fa=FA, dev=DEV):
    m = Model(V, T, D, VS, K_LOG, True, 'separate').to(dev)
    with torch.no_grad():
        m.layer1.W_q.weight.copy_(fa['Wq1']); m.layer1.W_k.weight.copy_(fa['Wk1'])
        m.layer1.W_v.weight.copy_(fa['Wv1']); m.layer1.W_o.weight.copy_(fa['Wo1'])
        m.layer2.W_q.weight.copy_(fa['Wq2']); m.layer2.W_k.weight.copy_(fa['Wk2'])
        m.layer2.W_v.weight.copy_(fa['Wv2']); m.layer2.W_o.weight.copy_(fa['Wo2'])
    return readout_(m, alpha, dev)

def build_combined(alpha, cb=CB, dev=DEV):
    m = Model(V, T, D, VS, K_LOG, True, 'combined').to(dev)
    with torch.no_grad():
        m.layer1.W_qk.copy_(cb['Wqk1']); m.layer1.W_ov.copy_(cb['Wov1'])
        m.layer2.W_qk.copy_(cb['Wqk2']); m.layer2.W_ov.copy_(cb['Wov2'])
    return readout_(m, alpha, dev)

msep, mcmb = build_separate(0.6), build_combined(0.6)
print(f'separate  {sum(p.numel() for p in msep.parameters()):,} parameters')
print(f'combined  {sum(p.numel() for p in mcmb.parameters()):,} parameters')
print(f'ff1/ff2/fc_out identical: '
      f'{all(torch.equal(getattr(msep, n).weight, getattr(mcmb, n).weight) for n in ("ff1","ff2","fc_out"))}')

separate  463,516 parameters
combined  428,524 parameters
ff1/ff2/fc_out identical: True


## 4. Equivalence

Four levels, from the matrices outward: the collapse itself, the residual stream after each
attention layer, the raw logits, and finally the cross-entropy on real data.

In [4]:
blob = torch.load(os.path.join(DATA, 'test', 'data_x_0.6.pt'), weights_only=False)
x = blob['x'][:256]

@torch.no_grad()
def streams(m, x):
    B, Tn = x.shape
    tok = F.one_hot(x, num_classes=V).float()
    pos = F.one_hot(torch.arange(Tn), num_classes=T).float().unsqueeze(0).expand(B, Tn, -1)
    s0 = torch.cat([tok, pos], -1)
    mask = torch.triu(torch.ones(Tn, Tn) * float('-inf'), diagonal=1)
    s1 = m.layer1(s0, mask)
    s2 = m.layer2(s1, mask)
    return s1, s2, m.fc_out(s2 + m.ff2(m.act(m.ff1(s2))))

s1a, s2a, za = streams(msep, x)
s1b, s2b, zb = streams(mcmb, x)
pa, pb = F.softmax(za[:, TS:TE], -1), F.softmax(zb[:, TS:TE], -1)

print('collapse residual (should be 0 by definition):')
for n, lhs, rhs in [('layer1 Wqk', CB['Wqk1'], FA['Wq1'].t() @ FA['Wk1']),
                    ('layer1 Wov', CB['Wov1'], FA['Wv1'].t() @ FA['Wo1'].t()),
                    ('layer2 Wqk', CB['Wqk2'], FA['Wq2'].t() @ FA['Wk2']),
                    ('layer2 Wov', CB['Wov2'], FA['Wv2'].t() @ FA['Wo2'].t())]:
    print(f'  {n:11s} max|diff| = {float((lhs - rhs).abs().max()):.3e}')

print('\nforward pass, separate vs combined:')
for n, A, Bv in [('stream after L1', s1a, s1b), ('stream after L2', s2a, s2b),
                 ('logits', za, zb), ('softmax probs', pa, pb)]:
    d = float((A - Bv).abs().max())
    rel = d / max(float(A.abs().max()), 1e-30)
    print(f'  {n:16s} max|diff| = {d:.3e}   relative = {rel:.3e}   '
          f'exactly equal: {torch.equal(A, Bv)}')

print(f'\ncount slots f_v identical: '
      f'{torch.equal(s2a[..., IN2:IN2+VS], s2b[..., IN2:IN2+VS])}')
print(f'argmax prediction identical on all {pa.shape[0] * pa.shape[1]} scored positions: '
      f'{bool((pa.argmax(-1) == pb.argmax(-1)).all())}')

collapse residual (should be 0 by definition):
  layer1 Wqk  max|diff| = 0.000e+00
  layer1 Wov  max|diff| = 0.000e+00
  layer2 Wqk  max|diff| = 0.000e+00
  layer2 Wov  max|diff| = 0.000e+00

forward pass, separate vs combined:
  stream after L1  max|diff| = 0.000e+00   relative = 0.000e+00   exactly equal: True
  stream after L2  max|diff| = 0.000e+00   relative = 0.000e+00   exactly equal: True
  logits           max|diff| = 0.000e+00   relative = 0.000e+00   exactly equal: True


  softmax probs    max|diff| = 0.000e+00   relative = 0.000e+00   exactly equal: True

count slots f_v identical: True
argmax prediction identical on all 12800 scored positions: True


## 5. Cross-entropy, every threshold

The only test that matters for downstream use: does the collapsed construction score the same?

In [5]:
@torch.no_grad()
def ce(m, blob, n_max=2048, batch=256):
    xs, ys = blob['x'][:n_max], blob['y'][:n_max]
    tot = n = 0.0
    for i in range(0, xs.shape[0], batch):
        xb = xs[i:i + batch]
        yb = ys[i:i + batch, TS:TE]
        lg = m(xb)[:, TS:TE]
        tot += F.cross_entropy(lg.reshape(-1, V), yb.reshape(-1), reduction='sum').item()
        n += yb.numel()
    return tot / n

THRS = [0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1.0]
print(f'{"alpha":>6s} {"separate":>12s} {"combined":>12s} {"difference":>13s}')
worst = 0.0
for a in THRS:
    b = torch.load(os.path.join(DATA, 'test', f'data_x_{a}.pt'), weights_only=False)
    ca, cb_ = ce(build_separate(a), b), ce(build_combined(a), b)
    worst = max(worst, abs(ca - cb_))
    print(f'{a:6.1f} {ca:12.6f} {cb_:12.6f} {ca - cb_:13.2e}')
print(f'\nworst CE difference across all thresholds: {worst:.3e} nats')

 alpha     separate     combined    difference


   0.4     1.842563     1.842563      0.00e+00


   0.5     1.791351     1.791351      0.00e+00


   0.6     1.696080     1.696080      0.00e+00


   0.7     1.561635     1.561635      0.00e+00


   0.8     1.382922     1.382922      0.00e+00


   0.9     1.137769     1.137769      0.00e+00


   1.0     0.746942     0.746942      0.00e+00

worst CE difference across all thresholds: 0.000e+00 nats


## 6. Verdict

In [6]:
ok_mat = all(float((CB[k] - v).abs().max()) == 0.0 for k, v in
             [('Wqk1', FA['Wq1'].t() @ FA['Wk1']), ('Wov1', FA['Wv1'].t() @ FA['Wo1'].t()),
              ('Wqk2', FA['Wq2'].t() @ FA['Wk2']), ('Wov2', FA['Wv2'].t() @ FA['Wo2'].t())])
ok_logit = torch.equal(za, zb)
ok_pred = bool((pa.argmax(-1) == pb.argmax(-1)).all())
ok_ce = worst < 1e-9

print(f'collapse is exact                       {ok_mat}')
print(f'logits bitwise identical                {ok_logit}')
print(f'predictions identical                   {ok_pred}')
print(f'CE identical to <1e-9 at every alpha    {ok_ce}   (worst {worst:.1e})')
print()
if ok_mat and ok_pred and ok_ce:
    print('YES — the eight hand-set factors can be multiplied into four circuit matrices and')
    print('used as the combined construction. The two parametrizations are the same function.')
    if not ok_logit:
        print()
        print('Note: the logits are not BITWISE identical. The two blocks multiply the same')
        print('matrices in a different order, so float32 rounding differs in the last places.')
        print('This is arithmetic, not a modelling difference — see the magnitudes above.')
else:
    print('MISMATCH — do not use the collapsed construction until this is understood.')

collapse is exact                       True
logits bitwise identical                True
predictions identical                   True
CE identical to <1e-9 at every alpha    True   (worst 0.0e+00)

YES — the eight hand-set factors can be multiplied into four circuit matrices and
used as the combined construction. The two parametrizations are the same function.
